# Part IV — the inverse problem: data-driven models

Companion notebook to `slides/part4.pdf`. It follows the slides in order:

1. **The inverse problem** — what we are given, what we want;
2. **Dynamic mode decomposition** — the dense baseline;
3. **Space-time MPS encoding** — space *and* time on one chain;
4. **A bonus PDE solver** — the all-at-once space-time solve;
5. **MPS-DMD** — DMD done inside the space-time train;
6. **TTs for DMD** — cost against the length of the time series, and what noise does;
7. **SINDy** — sparse identification of the equations themselves;
8. **SINDy's curse of dimensionality** and its tensor form;
9. **MANDy and more** — the dictionary as an exact tensor train.

Exercises are interleaved where the concept is introduced; each is followed directly by
its solution. Try the exercise first.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_dmd as dmd
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

from scipy.integrate import solve_ivp

## 1. The inverse problem

Parts I–III solved the *simulation* task $b = F(a)$: given the model $F$ (a PDE) and the
input conditions $a$, compute $b$. The *inverse* problem asks for $a$ given $b$. Here we
go one step further and **discover the model**: given pairs $\{(a_i, b_i)\}_{i=1}^M$,

$$\hat F = \arg\min_{G\in\mathcal M} \sum_i \lVert b_i - G(a_i)\rVert^2 .$$

For dynamics the data is a time series,

$$\mathcal D = \{\mathbf u_k\}_{k=0}^{N_t}, \qquad \mathbf u_k = \mathbf u(t_k)\in\mathbb R^{N_x},
\qquad t_k = k\,\Delta t,$$

and $\frac{d\mathbf u}{dt} = \mathbf f(\mathbf u)$ becomes the map $\mathbf u_{k+1} = \mathbf F(\mathbf u_k)$, fitted by

$$\hat{\mathbf F} = \arg\min_{\mathbf G\in\mathcal M}\sum_{k=0}^{N_t-1}
\lVert \mathbf u_{k+1} - \mathbf G(\mathbf u_k)\rVert_2^2 .$$

Two tasks, kept apart because methods strong at one are often weak at the other:

| task | goal |
|---|---|
| **prediction** | future states $\hat{\mathbf u}_{N_t+k} = \hat{\mathbf F}^{\,k}(\mathbf u_{N_t})$ |
| **learning** | the spectrum: $\hat{\mathbf F}(\mathbf u) = A\mathbf u$, $\;A\boldsymbol\phi_j = \lambda_j\boldsymbol\phi_j$ |

On the spectrum from fully data-driven (neural networks) to fully physics-driven (solving
PDEs), DMD and SINDy sit in between: they learn from data, but return a model you can read.

A dataset with a known answer: a travelling wave, a slower travelling wave, and a decaying
structure. Three spatial structures, so five eigenvalues, and we know exactly what they
should be.

In [ ]:
n_x, n_t = 10, 9
N_x, N_t = 2 ** n_x, 2 ** n_t
x = np.linspace(0, 1, N_x, endpoint=False)
dt = 0.004
t = np.arange(N_t) * dt
c1, c2, decay = 1.0, -0.4, -0.3

X = (np.sin(2 * np.pi * (x[:, None] - c1 * t[None, :]))
     + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * t[None, :]))
     + 0.5 * np.exp(decay * t)[None, :] * np.cos(10 * np.pi * x)[:, None])

expected = np.sort_complex(np.array([
    np.exp(2j * np.pi * c1 * dt), np.exp(-2j * np.pi * c1 * dt),
    np.exp(6j * np.pi * c2 * dt), np.exp(-6j * np.pi * c2 * dt),
    np.exp(decay * dt)]))

plt.imshow(X[:, ::8].T, aspect="auto", origin="lower", cmap="RdBu_r",
           extent=[0, 1, 0, t[-1]])
plt.xlabel("x"), plt.ylabel("t"), plt.title("the data")
plt.tight_layout()
print(f"snapshot matrix: {N_x:,d} x {N_t:,d} = {X.size:,d} numbers")

## 2. Dynamic mode decomposition

Collect the snapshots into two shifted matrices and regress a linear map between them:

$$X = [\mathbf u_0, \dots, \mathbf u_{N_t-1}], \qquad X' = [\mathbf u_1, \dots, \mathbf u_{N_t}],
\qquad X' = AX \;\Rightarrow\; A = X'X^{+}.$$

In practice $A$ is never formed. With $X = U\Sigma V^\dagger$,

$$\tilde A = U^\dagger X' V \Sigma^{-1}, \quad \tilde A W = W\Lambda, \quad \Phi = UW .$$

Each eigenvalue gives a growth rate and a frequency; each eigenvector a spatial mode. The
bottleneck is the SVD of an $N_x \times N_t$ matrix, prohibitive long before the physics is.

In [ ]:
t0 = time.perf_counter()
ref = dmd.exact_dmd(X, rank=5)
t_exact = time.perf_counter() - t0
print(f"exact DMD: {t_exact:.3f} s")
print("eigenvalues :", np.sort_complex(ref["eigenvalues"]))
print("expected    :", expected)

## 3. Space-time MPS encoding

**#1. Discretisation.** Binarise **both** coordinates,

$$x = \sum_{k=1}^{n} i_k\,2^{-k}, \qquad t = T\sum_{l=1}^{m} j_l\,2^{-l}, \qquad i_k, j_l\in\{0,1\},$$

so $f(x,t) \leftrightarrow f(i_1,\dots,i_n,j_1,\dots,j_m)$.

**#2. MPS encoding.** Put all the bits on one chain, space first then time:
$f(x,t) \to A_1^{(i_1)}\cdots A_n^{(i_n)}A_{n+1}^{(j_1)}\cdots A_{n+m}^{(j_m)}$.
Time stops being a loop and becomes more legs.

In [ ]:
cores = dmd.space_time_qtt(X, eps=1e-10)
print(f"sites        : {len(cores)}  ({n_x} space + {n_t} time)")
print(f"bond dims    : {tn.tt_ranks(cores)}")
print(f"parameters   : {tn.tt_size(cores):,d}  vs  {X.size:,d} dense "
      f"({X.size / tn.tt_size(cores):.0f}x)")
print(f"space/time bond: {tn.tt_ranks(cores)[n_x]}")

That last number is not a bookkeeping detail. **The bond at the space/time interface is
the temporal complexity of the dataset**, read off directly. Compare a travelling wave
with a standing wave:

In [ ]:
for name, data in [
    ("travelling  sin(2pi(x - ct))", np.sin(2 * np.pi * (x[:, None] - t[None, :]))),
    ("standing    sin(2pi x) cos(2pi t)",
     np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :]),
    ("three modes (the data above)", X),
    ("white noise", np.random.default_rng(0).standard_normal((N_x, N_t))),
]:
    c = dmd.space_time_qtt(data, eps=1e-10)
    print(f"{name:>34}: space/time bond = {tn.tt_ranks(c)[n_x]:>4}")

A standing wave is a single product $f(x)g(t)$, so it is rank 1 across that cut.
A travelling wave is a sum of two, so rank 2. Noise has no structure to share and the
bond saturates. You have measured "how many degrees of freedom does this dataset have
in time" without running any algorithm.

---
### Exercise 1 — read the temporal complexity off the bond

**Point:** the bond at the space/time interface counts the temporal degrees of freedom,
and you get it without running any algorithm.

Write down your prediction for each dataset below, then measure with
`tn.tt_ranks(dmd.space_time_qtt(data))[n_x]`.

1. $\sin(2\pi x)\cos(2\pi t)$
2. $\sin(2\pi(x - t))$
3. a sum of $m$ travelling waves with distinct speeds, for $m = 1, 2, 3, 4$
4. $\sin(2\pi x)\,\cos(2\pi t)$ plus noise of amplitude $10^{-3}$, at tolerances
   $10^{-10}$ and $10^{-2}$
5. a travelling *front*, $\tanh(20(x - 0.3 - 0.5t))$

Item 4 is the one to think hard about. What does the tolerance have to do with how
complicated the data *is*?

In [ ]:
# TODO

#### Solution to Exercise 1

*Try the exercise before reading on.*

In [ ]:
# --- Exercise 1 ---
rng = np.random.default_rng(0)
cases = {
    "standing sin(x)cos(t)": np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :],
    "travelling sin(x - t)": np.sin(2 * np.pi * (x[:, None] - t[None, :])),
    "2 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4)),
    "3 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4, 2.2)),
    "4 travelling waves": sum(np.sin(2 * np.pi * (x[:, None] - c * t[None, :]))
                              for c in (1.0, -0.4, 2.2, 0.7)),
    "travelling front (tanh)": np.tanh(20 * (x[:, None] - 0.3 - 0.5 * t[None, :])),
}
for name, data in cases.items():
    c = dmd.space_time_qtt(data, eps=1e-10)
    print(f"{name:>26}: space/time bond = {tn.tt_ranks(c)[n_x]:>4}")

noisy = (np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :]
         + 1e-3 * rng.standard_normal((N_x, N_t)))
for eps in (1e-10, 1e-2):
    c = dmd.space_time_qtt(noisy, eps=eps)
    print(f"{'standing + 1e-3 noise':>26}: space/time bond = "
          f"{tn.tt_ranks(c)[n_x]:>4}  (tolerance {eps:.0e})")
print("""
Each travelling wave contributes 2 to the bond (it is a sum of two products), a standing
wave contributes 1, and the tanh front is not a finite sum of products at all -- its bond
is set by how accurately you insist on representing it. The noise case makes the point
sharply: at 1e-10 the bond saturates, because you have demanded that the noise be
reproduced exactly; at 1e-2 it collapses back to 1. The bond measures complexity *at the
accuracy you asked for*, which is the only kind of complexity there is.""")

## 4. A bonus PDE solver

For $\partial_t u + F(u, \partial_x u, \dots, \partial_x^q u) = g(x,t)$ there are two ways to
use the chain.

* **Time-stepping:** $\mathbf u_{t+1} = \mathbf u_t + \mathbb F(\mathbf u_t)\,\Delta t$ — one spatial
  MPS, updated $N_t$ times (Part III).
* **Space-time methods:** assemble the whole history as one linear system on the
  space-time chain,
  $$(\mathbb I_x \otimes \mathbb D_t + \mathbb F_x \otimes \mathbb I_t)\,\mathbf u = \text{b.c.} + \text{i.c.},$$
  and solve it once. A nonlinear $F$ requires Newton iterations with the space-time Jacobian.

The space-time train that comes out of such a solve is exactly the input MPS-DMD wants.

---
### Exercise 2 — solve a PDE all at once, with no time loop

**Point:** if space and time are just sites on one chain, "time stepping" is only one
way to solve the system, and not obviously the best one.

Assemble the entire space-time system for the heat equation as a single MPO and solve
it with one DMRG call. Backward Euler on a space-time grid reads

$$\frac{U_k - U_{k-1}}{\Delta\tau} - \alpha L_x U_k = f_k,
  \qquad U_{-1} \equiv 0, \qquad f_0 = \frac{u_0}{\Delta\tau},$$

which is one linear system $M\,U = b$ over $n_x + n_t$ sites.

Steps:

1. build $D_t = (\mathbb{1} - S_t)/\Delta\tau$ on the time sites, with $S_t$ the
   non-periodic shift — its "missing" first column is exactly the $U_{-1}=0$ condition;
2. build $M = \mathbb{1}_x \otimes D_t - \alpha L_x \otimes \mathbb{1}_t$ (so $\mathbb F_x = -\alpha L_x$ in the slide's notation);
3. build $b = (u_0/\Delta\tau) \otimes \delta_{t,0}$, where $\delta_{t,0}$ is the rank-1
   train with cores $(1, 0)$;
4. $M$ is not symmetric and `dmrg_solve` wants an SPD operator, so solve the normal
   equations $M^\top M\,U = M^\top b$. Transposing an MPO is
   `c.transpose(0, 2, 1, 3)` on every core;
5. check the answer against the backward-Euler recursion you never ran.

Once it works, the interesting part: swap $u_0$ for something non-separable in
space-time — a narrow Gaussian bump, say — and watch the space/time bond dimension.
That number is what an all-at-once method has to pay, and it is what decides whether
removing the time loop was a good idea.

In [ ]:
nx, nt = 8, 6
Nx, Nt = 2 ** nx, 2 ** nt
h = 2.0 ** -nx
alpha = 1.0
T = 0.002
dtau = T / Nt
v0 = np.sin(np.pi * (np.arange(Nx) + 1) / (Nx + 1))          # a discrete eigenvector
lam = -4.0 / h ** 2 * np.sin(np.pi / (2 * (Nx + 1))) ** 2     # its eigenvalue

# TODO

#### Solution to Exercise 2

*Try the exercise before reading on.*

In [ ]:
# --- Exercise 2 ---
def eye_mpo(k):
    return [np.eye(2).reshape(1, 2, 2, 1) for _ in range(k)]


Lx = tn.qtt_laplacian(nx, dx=h)
St = tn.qtt_shift(nt, +1)                                   # (S u)[k] = u[k-1]
Dt = tn.mpo_scale(
    tn.mpo_round(tn.mpo_add(tn.mpo_identity(nt), tn.mpo_scale(St, -1.0)), 1e-13),
    1.0 / dtau)

M = tn.mpo_round(tn.mpo_add(eye_mpo(nx) + Dt,
                            tn.mpo_scale(Lx + eye_mpo(nt), -alpha)), 1e-12)
MT = [c.transpose(0, 2, 1, 3).copy() for c in M]
A = tn.mpo_round(tn.mpo_compose(MT, M), 1e-12)

delta0 = [np.array([1.0, 0.0]).reshape(1, 2, 1) for _ in range(nt)]
b = tn.tt_round(tn.qtt_from_vector(v0 / dtau, eps=1e-13) + delta0, 1e-13)
rhs = tn.tt_round(tn.mpo_apply(MT, b), 1e-12)

t0 = time.perf_counter()
U = tn.dmrg_solve(A, rhs, sweeps=6, eps=1e-10, chi_max=40)
print(f"M rank {max(tn.mpo_ranks(M))}, M^T M rank {max(tn.mpo_ranks(A))}")
print(f"one solve for ALL {Nt} timesteps: {time.perf_counter() - t0:.2f} s, "
      f"chi = {max(tn.tt_ranks(U))}, residual {tn.residual(A, U, rhs):.1e}")

full = tn.tt_full(U).reshape(Nx, Nt)
for k in (0, Nt // 2, Nt - 1):
    be = v0 / (1 - lam * dtau) ** (k + 1)                   # backward Euler in closed form
    print(f"  t index {k:>2}: error vs backward Euler "
          f"{np.linalg.norm(full[:, k] - be) / np.linalg.norm(be):.2e}")

In [ ]:
# --- Exercise 2, the interesting variant: a non-separable initial condition ---
xg = np.linspace(0, 1, Nx, endpoint=False)
for label, u0 in [("eigenvector (separable in space-time)", v0),
                  ("narrow gaussian bump", np.exp(-((xg - 0.5) / 0.02) ** 2))]:
    b2 = tn.tt_round(tn.qtt_from_vector(u0 / dtau, eps=1e-13) + delta0, 1e-13)
    rhs2 = tn.tt_round(tn.mpo_apply(MT, b2), 1e-12)
    U2 = tn.dmrg_solve(A, rhs2, sweeps=6, eps=1e-10, chi_max=60)
    print(f"{label:>38}: space/time bond = {tn.tt_ranks(U2)[nx]:>3}, "
          f"max chi = {max(tn.tt_ranks(U2)):>3}")
print("""
The separable case costs a bond of 1 or 2; the Gaussian costs more, because the diffusion
of a localised bump is not a short sum of products of a function of x and a function of t.
That bond is the price of removing the time loop, and it is exactly what the space-time
papers spend their effort bounding.""")

## 5. MPS-DMD

Same algorithm as Section 2, but every step happens inside the space-time train.

1. **POD of $X$.** Gauge the train to mixed canonical form at the space/time bond. The
   Schmidt decomposition across that bond **is** $X = U\Sigma V^\dagger$ — the SVD that
   standard DMD spends its whole budget computing. Then $A = X'X^{+} = X'V\Sigma^{-1}U^\dagger$.
2. **Project $A$ onto the span of $U$:** $\tilde A := U^\dagger A U = U^\dagger X' V\Sigma^{-1}$.
   $X'$ is the time-shifted train (a rank-2 shift MPO on the time sites), so $\tilde A$ is a
   $\chi\times\chi$ contraction.
3. **Eigendecomposition:** $\tilde A W = W\Lambda \Rightarrow A\,UW = UW\,\Lambda$; modes $\Phi = UW$,
   still in train form.
4. **Prediction:** $\mathbf u_{T+k} = A^k\mathbf u_T$ with $A^k = UW\Lambda^kW^{-1}U^\dagger$.

No snapshot matrix is formed at any point.

In [ ]:
t0 = time.perf_counter()
res = dmd.mps_dmd(cores, n_x, rank=5)
t_mps = time.perf_counter() - t0
print(f"MPS-DMD (given the train): {t_mps * 1e3:.1f} ms")
print("eigenvalues :", np.sort_complex(res["eigenvalues"]))
print("exact DMD   :", np.sort_complex(ref["eigenvalues"]))
print(f"\nmax difference: "
      f"{np.abs(np.sort_complex(res['eigenvalues']) - np.sort_complex(ref['eigenvalues'])).max():.2e}")

Identical to machine precision. Same modes, same eigenvalues, same predictions —
obtained from a $\chi\times\chi$ matrix instead of an $N_x\times N_t$ one.

### The one subtlety worth stating out loud

The non-periodic time shift sends the final snapshot to zero, so the raw projection is
$QP^\top$ rather than the least-squares $QP^{+}$ that DMD wants. Writing $v$ for the
last column of $V^\dagger$, $PP^\top = I - vv^\top$, so one Sherman–Morrison rank-one
correction recovers exact DMD. Skip it and every eigenvalue is damped by roughly
$1/N_t$ per step — invisible in the spectrum, ruinous after a few hundred steps.

In [ ]:
# What the uncorrected version would give:
g = tn.tt_canonicalise(cores, centre=n_x - 1)
c0 = g[n_x - 1]
r, d, r1 = c0.shape
u_, s_, vt_ = np.linalg.svd(c0.reshape(r * d, r1), full_matrices=False)
k = 5
u_, s_, vt_ = u_[:, :k], s_[:k], vt_[:k]
right = [np.tensordot(vt_, g[n_x], axes=([1], [0]))] + g[n_x + 1:]
shifted = tn.tt_round(tn.mpo_apply(tn.qtt_shift(n_t, -1), right), eps=1e-12)
M = dmd._overlap_matrix(shifted, right)
lam_raw = np.linalg.eigvals((s_[:, None] * M) / s_[None, :])
print("uncorrected |lambda|:", np.sort(np.abs(lam_raw)))
print("corrected   |lambda|:", np.sort(np.abs(res["eigenvalues"])))
print("true        |lambda|:", np.sort(np.abs(expected)))
print(f"\nuncorrected damping per step: {1 - np.abs(lam_raw).max():.2e}  "
      f"(compare 1/N_t = {1 / N_t:.2e})")

### Step 4: prediction

$u(T+k) = U W \Lambda^k W^{-1} U^\dagger u_T$: one diagonal multiply and one contraction
of the open bond. The cost does not depend on how far ahead you predict.

In [ ]:
future = [0, N_t // 2, N_t - 1, N_t + 500, N_t + 2000]
print(f"{'step':>8} {'relative error':>16}  {'':>4}")
for k in future:
    pred = dmd.predict_dense(res, k)
    truth = (np.sin(2 * np.pi * (x - c1 * k * dt))
             + 0.6 * np.cos(6 * np.pi * (x - c2 * k * dt))
             + 0.5 * np.exp(decay * k * dt) * np.cos(10 * np.pi * x))
    tag = "within data" if k < N_t else "extrapolated"
    print(f"{k:>8,d} {np.linalg.norm(pred - truth) / np.linalg.norm(truth):>16.2e}  {tag}")

plt.plot(x, dmd.predict_dense(res, N_t + 2000), label="MPS-DMD, 2000 steps ahead")
k = N_t + 2000
plt.plot(x, (np.sin(2 * np.pi * (x - c1 * k * dt))
             + 0.6 * np.cos(6 * np.pi * (x - c2 * k * dt))
             + 0.5 * np.exp(decay * k * dt) * np.cos(10 * np.pi * x)),
         "--", label="truth")
plt.legend(), plt.xlabel("x")
plt.tight_layout()

It extrapolates because it identified the *operator*, not because it memorised the
data. That is the difference between the two tasks in the table at the top, and it is
also the honest limit of the method: if the true dynamics are not close to linear in
the observables you gave it, no amount of data will rescue the extrapolation.

---
### Exercise 3 — break MPS-DMD on purpose

**Point:** DMD is a linear model in the observables you give it, and the failure is
structural, not numerical.

Apply `dmd.mps_dmd` to each of these and look at what comes back:

1. pure exponential growth, $e^{0.5t}\sin(2\pi x)$ — should be exact;
2. a **standing** wave, $\sin(2\pi x)\cos(2\pi t)$ — try rank 1 and rank 2 and explain
   what you see;
3. a frequency-swept wave, $\sin(2\pi(x - t(1 + 0.5t)))$;
4. a nonlinearity, $u(x,t) = \tanh\!\big(3\sin(2\pi x)\,e^{t}\big)$.

For each, report the eigenvalues and — most usefully — whether the *reconstruction
within* the data was any good.

Case 2 is the classic trap: a standing wave has spatial rank 1, so no DMD can find the
two eigenvalues $e^{\pm 2\pi i \Delta t}$ that generate it. The fix is a time-delay
(Hankel) embedding — which, in the space-time train, is a statement about which sites
you cut. Worth a discussion even if you do not implement it.

In [ ]:
# TODO

#### Solution to Exercise 3

*Try the exercise before reading on.*

In [ ]:
# --- Exercise 3 ---
def try_dmd(name, data, rank):
    c = dmd.space_time_qtt(data, eps=1e-10)
    r = dmd.mps_dmd(c, n_x, rank=rank)
    recon = np.array([dmd.predict_dense(r, k) for k in range(0, N_t, N_t // 8)]).T
    ref = data[:, ::N_t // 8]
    err_in = np.linalg.norm(recon - ref) / np.linalg.norm(ref)
    print(f"{name:>24} rank {rank}: |lambda| = "
          f"{np.round(np.sort(np.abs(r['eigenvalues'])), 5)},  "
          f"in-sample error {err_in:.2e}")


try_dmd("exponential growth",
        np.exp(0.5 * t)[None, :] * np.sin(2 * np.pi * x)[:, None], 1)
try_dmd("standing wave",
        np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :], 1)
try_dmd("standing wave",
        np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :], 2)
try_dmd("swept frequency",
        np.sin(2 * np.pi * (x[:, None] - t[None, :] * (1 + 0.5 * t[None, :]))), 6)
try_dmd("tanh nonlinearity",
        np.tanh(3 * np.sin(2 * np.pi * x)[:, None] * np.exp(t)[None, :]), 6)
print("""
The exponential is exact: it genuinely is a one-mode linear system. The standing wave has
spatial rank 1, so DMD can only ever return ONE eigenvalue, and asking for rank 2 does not
help -- the two frequencies that generate cos(2 pi t) are simply not visible in a
one-dimensional column space. The swept and tanh cases fit the in-sample data tolerably
and mean nothing outside it, because the dynamics are not linear in the observables
supplied. Note that DMD does not fail loudly in any of these. You have to check.""")

## 6. TTs for DMD: cost, and what noise does

Earlier tensor-based DMD (Klus et al. 2018) compressed space but left time uncompressed,
so it stays linear in $N_t$; Środa et al. reach the same complexity independently. With
both space and time on the chain,

$$\text{DMD: } \mathcal O\big(N_xN_t\min(N_x,N_t)\big), \qquad
\text{MPS-DMD: } \mathcal O\big((n+m)\chi^3\big), \quad N_x = 2^n,\; N_t = 2^m .$$

In [ ]:
print(f"{'N_t':>8} {'exact DMD (s)':>15} {'MPS-DMD (ms)':>14} {'chi':>5}")
for nt in (6, 8, 10, 12):
    Nt = 2 ** nt
    tt = np.arange(Nt) * dt
    data = (np.sin(2 * np.pi * (x[:, None] - c1 * tt[None, :]))
            + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * tt[None, :]))
            + 0.5 * np.exp(decay * tt)[None, :] * np.cos(10 * np.pi * x)[:, None])
    c = dmd.space_time_qtt(data, eps=1e-10)

    t0 = time.perf_counter()
    dmd.exact_dmd(data, rank=5)
    te = time.perf_counter() - t0

    t0 = time.perf_counter()
    r_ = dmd.mps_dmd(c, n_x, rank=5)
    tm = time.perf_counter() - t0
    print(f"{Nt:>8,d} {te:>15.3f} {tm * 1e3:>14.1f} {r_['rank']:>5}")

Two honest caveats on that table.

* The MPS-DMD column excludes building the space-time train. That is the right
  comparison when the train came *out of* a space-time solver, which is the intended
  setting. For raw external data you stack snapshot trains one at a time and truncate
  after each — linear in the number of snapshots, and done once.
* The exact-DMD column is dominated by LAPACK, which is very good. The crossover is a
  matter of problem size, not of one method being cleverer.

### What noise does

Truncation looks like denoising. Sometimes it is. There is no theory saying when.

In [ ]:
rng = np.random.default_rng(5)
clean = X
print(f"{'noise level':>12} {'chi (eps=1e-2)':>16} {'eig error, no trunc':>22} "
      f"{'eig error, chi=5':>18}")
for sigma in (0.0, 0.01, 0.05, 0.2):
    noisy = clean + sigma * rng.standard_normal(clean.shape)
    c_loose = dmd.space_time_qtt(noisy, eps=1e-2)
    r_full = dmd.mps_dmd(dmd.space_time_qtt(noisy, eps=1e-10), n_x, rank=5)
    r_trunc = dmd.mps_dmd(dmd.space_time_qtt(noisy, eps=1e-10, chi_max=5), n_x, rank=5)

    def eig_err(r):
        got = np.sort_complex(r["eigenvalues"])
        return np.abs(got - expected).max()

    print(f"{sigma:>12.2f} {max(tn.tt_ranks(c_loose)):>16} "
          f"{eig_err(r_full):>22.2e} {eig_err(r_trunc):>18.2e}")

Truncating to the true rank helps: the noise lives in the discarded singular values.
But the cross-over is empirical, the "true rank" is exactly what you did not know, and
nothing in the output tells you whether you truncated the noise or the physics. This is
an open problem, not a solved one — as are uncertainty quantification on tensor-network
predictions and the extension of this analysis beyond one spatial dimension.

What tensor networks *do* have over a neural surrogate here is that the answer comes out
as modes, frequencies and growth rates. You can argue with it.

---
### Exercise 4 — truncation as a denoiser

**Point:** it often works, nothing tells you when, and that is an open problem rather
than a technique.

Add Gaussian noise of amplitude $\sigma$ to the three-mode dataset and sweep the
truncation rank. For each $(\sigma, \chi)$ record the error in the recovered
eigenvalues. Plot the error against $\chi$ for each $\sigma$ and find the optimum.

Then the question that matters: **in a real problem you do not have `expected`.** What
would you monitor instead, and how confident should you be?

In [ ]:
clean = (np.sin(2 * np.pi * (x[:, None] - c1 * t[None, :]))
         + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * t[None, :]))
         + 0.5 * np.exp(decay * t)[None, :] * np.cos(10 * np.pi * x)[:, None])
# (`expected` is defined in Section 1)

# TODO

#### Solution to Exercise 4

*Try the exercise before reading on.*

In [ ]:
# --- Exercise 4 ---
plt.figure(figsize=(8, 3.5))
for sigma in (0.01, 0.05, 0.2):
    noisy = clean + sigma * rng.standard_normal(clean.shape)
    chis, errs = [], []
    for chi in (3, 5, 8, 12, 20, 40, 80):
        c = dmd.space_time_qtt(noisy, eps=1e-12, chi_max=chi)
        r = dmd.mps_dmd(c, n_x, rank=5)
        lam_got = np.sort_complex(r["eigenvalues"])
        if len(lam_got) < 5:
            lam_got = np.pad(lam_got, (0, 5 - len(lam_got)))
        chis.append(chi)
        errs.append(np.abs(lam_got - expected).max())
    plt.loglog(chis, errs, "o-", label=f"$\\sigma={sigma}$")
    print(f"sigma = {sigma:>4}: best chi = {chis[int(np.argmin(errs))]:>3}, "
          f"eigenvalue error {min(errs):.2e}")
plt.xlabel("truncation rank $\\chi$"), plt.ylabel("max eigenvalue error")
plt.legend(), plt.tight_layout()
print("""
Read the optimum as a function of noise. At sigma = 0.01 the error keeps falling as chi
grows: the noise is small enough that keeping a little of it costs less than discarding
signal. At sigma = 0.05 and above the optimum drops back to about the true rank, and
over-truncating and under-truncating both hurt. In a real problem you have no
`expected` to plot against. What you can monitor is the singular-value spectrum at the
space/time bond, looking for a knee; the stability of the eigenvalues as chi varies; and
whether the modes are physically plausible. None of these is a guarantee, and error bars
on tensor-network predictions are essentially an open problem.""")

## 7. SINDy

DMD recovers an operator, tied to the regime it was fitted in. SINDy (Brunton, Proctor &
Kutz 2016) goes for the **equation**. For $\dot{\mathbf x}(t) = \mathbf f(\mathbf x(t))$, stack the
states and their derivatives row by row (time down, state across),

$$\mathbf X = \begin{bmatrix}\mathbf x^T(t_1)\\ \vdots\\ \mathbf x^T(t_m)\end{bmatrix}, \qquad
\dot{\mathbf X} = \begin{bmatrix}\dot{\mathbf x}^T(t_1)\\ \vdots\\ \dot{\mathbf x}^T(t_m)\end{bmatrix},$$

evaluate a library of candidate functions $\Theta(\mathbf X)$ (constant, $x$, $y$, $xy$, ...), and
solve $\dot{\mathbf X} = \Theta(\mathbf X)\,\Xi$ for a **sparse** coefficient matrix $\Xi$ by
sequentially thresholded least squares. The nonzero entries of $\Xi$ *are* the recovered
equations.

The Lorenz system, with a library of all monomials up to degree 2:

In [ ]:
def lorenz(_, s, sigma=10.0, rho=28.0, beta=8 / 3):
    x_, y_, z_ = s
    return [sigma * (y_ - x_), x_ * (rho - z_) - y_, x_ * y_ - beta * z_]


dt_l = 1e-3
t_l = np.arange(0, 20, dt_l)
sol = solve_ivp(lorenz, (0, t_l[-1]), [-8.0, 8.0, 27.0], t_eval=t_l, rtol=1e-12, atol=1e-12)
Xl = sol.y.T                                           # rows = time, columns = state
dXl = np.gradient(Xl, dt_l, axis=0)                    # derivatives from the data alone

names = ["1", "x", "y", "z", "xx", "xy", "xz", "yy", "yz", "zz"]
xs, ys, zs = Xl.T
Theta = np.column_stack([np.ones_like(xs), xs, ys, zs,
                         xs * xs, xs * ys, xs * zs, ys * ys, ys * zs, zs * zs])


def stlsq(Theta, dX, threshold=0.1, iters=10):
    """Sequentially thresholded least squares (Brunton et al. 2016)."""
    Xi = np.linalg.lstsq(Theta, dX, rcond=None)[0]
    for _ in range(iters):
        small = np.abs(Xi) < threshold
        Xi[small] = 0
        for k in range(dX.shape[1]):
            keep = ~small[:, k]
            Xi[keep, k] = np.linalg.lstsq(Theta[:, keep], dX[:, k], rcond=None)[0]
    return Xi


Xi = stlsq(Theta, dXl)
for k, lhs in enumerate("xyz"):
    terms = " ".join(f"{c:+.3f}{'' if n == '1' else ' ' + n}"
                     for c, n in zip(Xi[:, k], names) if c != 0)
    print(f"d{lhs}/dt = {terms}")

Exactly the Lorenz equations ($\sigma = 10$, $\rho = 28$, $\beta = 8/3$), out of ten
candidates per component — and the output is an equation, not a black box.

## 8. SINDy: curse of dimensionality, and the tensor form

From here on we follow the slides' transposed convention: data $\mathbf X\in\mathbb R^{N\times M}$
with $\mathbf X[i,m]\approx x_i(t_m)$, $N$ the state dimension and $M$ the number of samples.
Pick $J$ univariate functions $\psi_1,\dots,\psi_J$ and take **all products** as the library,

$$\phi_{\mathbf j}(\mathbf x) = \prod_{i=1}^N \psi_{j_i}(x_i), \qquad
\Theta_\flat[\mathbf j, m] = \phi_{\mathbf j}(\mathbf X[:,m]), \qquad
\min_\Xi \big\lVert\dot{\mathbf X} - \Xi\,\Theta_\flat(\mathbf X)\big\rVert_F^2 + \mathcal R(\Xi).$$

That library is complete and needs no guessing which cross terms matter — but $\Theta_\flat$
is $J^N\times M$ and $\Xi$ is $N\times J^N$: exponential in the state dimension. In tensor
form $\Xi$ and $\Theta$ share $N$ legs of size $J$, a sum over $J^N$ terms.

In [ ]:
M_samples = 1000
for N in (3, 10, 20, 50):
    J = 4
    print(f"N = {N:>2}, J = {J}: Theta has {float(J) ** N * M_samples:9.2e} entries "
          f"({float(J) ** N * M_samples * 8 / 1e9:9.2e} GB)")

## 9. MANDy and more

The way out (Gelß et al. 2019, *MANDy*): $\Theta_\flat$ is a Khatri–Rao product,
$\Theta_\flat = \Psi_1\odot\cdots\odot\Psi_N$ with $\Psi_i[j,m] = \psi_j(\mathbf X[i,m])$, so it is an
**exact tensor train** whose cores are the small matrices $\Psi_i$ and whose bonds are the
sample index $m$ (size $M$). Orthonormalise that train and its last core gives
$\Theta_\flat = U\Sigma V^\top$ with $U = U_1\cdots U_N$ in train form. The least-squares
coefficients then come in closed form, also as a train:

$$\Xi = \dot{\mathbf X}\,\Theta_\flat^{+} = \big(\dot{\mathbf X}V\Sigma^{-1}\big)\,U^\top .$$

No $J^N$ object is ever formed. (Goeßmann et al. 2020 go further: $\Xi$ itself as a
low-rank TT plus a selection tensor, learned by ALS.)

---
### Exercise 5 — MANDy on a Fermi–Pasta–Ulam chain

**Point:** a dictionary of $J^N$ functions is affordable when it is a tensor train.

The FPU chain of $N$ oscillators with fixed ends ($x_0 = x_{N+1} = 0$) has

$$\ddot x_i = (x_{i+1} - 2x_i + x_{i-1}) + \beta\big[(x_{i+1}-x_i)^3 - (x_i-x_{i-1})^3\big],$$

a second-order system, so the "derivative data" is the acceleration $\ddot{\mathbf X}$. Use
$\psi \in \{1, x, x^2, x^3\}$ ($J = 4$) — the true right-hand side lies in that product
library.

1. Write `mandy(X, Y)`: sweep left to right through the cores $\Psi_i$ (contract the
   carried $M$-bond, QR, carry $R$ on), and SVD the last one to get $U_N$, $\Sigma$, $V$.
   Return the cores $U_1,\dots,U_N$ and the small matrix $Y V\Sigma^{-1}$.
2. Write `mandy_eval(cores, last, X_test)` that contracts $\Xi$ with
   $\theta(\mathbf x) = \otimes_i\psi(x_i)$ for each test point — a rank-1 train, so one cheap sweep.
3. Check against dense least squares for $N = 4$ ($J^N = 256$), then run $N = 8$
   ($J^N = 65\,536$ candidate functions) and report the test error on fresh samples.

In [ ]:
def psi(X):                                             # (N, M) -> (N, J, M)
    return np.stack([np.ones_like(X), X, X ** 2, X ** 3], axis=1)


def fpu(X, beta=0.7):
    """Accelerations of the FPU chain with fixed ends, column-wise."""
    pad = np.zeros((1, X.shape[1]))
    Xp, Xm = np.vstack([X[1:], pad]), np.vstack([pad, X[:-1]])
    return (Xp - 2 * X + Xm) + beta * ((Xp - X) ** 3 - (X - Xm) ** 3)


rng_f = np.random.default_rng(1)

# TODO: mandy(X, Y) and mandy_eval(cores, last, X_test)

#### Solution to Exercise 5

*Try the exercise before reading on.*

In [ ]:
def mandy(X, Y, eps=1e-10):
    """Xi = Y Theta^+ with Theta = Psi_1 (.) ... (.) Psi_N, as a train. Never forms J^N."""
    P = psi(X)
    N, J, M = P.shape
    cores, R = [], np.ones((1, M))                      # R: the part not yet orthonormalised
    for i in range(N):
        c = R[:, None, :] * P[i][None, :, :]            # core i is diagonal in M: (r, J, M)
        r = c.shape[0]
        if i < N - 1:
            q, R = np.linalg.qr(c.reshape(r * J, M))
            cores.append(q.reshape(r, J, -1))
        else:                                           # last core: Theta = U Sigma V^T
            u, s, vt = np.linalg.svd(c.reshape(r * J, M), full_matrices=False)
            k = int((s > eps * s[0]).sum())
            cores.append(u[:, :k].reshape(r, J, k))
            last = (Y @ vt[:k].T) / s[:k]               # Y V Sigma^{-1}
    return cores, last


def mandy_eval(cores, last, X_test):
    """Xi theta(x) for every column x of X_test: one sweep with a rank-1 train."""
    P = psi(X_test)                                     # (N, J, P)
    v = np.ones((X_test.shape[1], 1))
    for c, p in zip(cores, P):
        v = np.einsum("pa,ajb,jp->pb", v, c, p)
    return last @ v.T


# check against dense least squares for N = 4
N = 4
X4 = rng_f.uniform(-0.1, 0.1, (N, 400))
P4 = psi(X4)
Theta_dense = P4[0]
for i in range(1, N):                                   # Khatri-Rao: J^N x M
    Theta_dense = np.einsum("am,jm->ajm", Theta_dense, P4[i]).reshape(-1, X4.shape[1])
Xi_dense = fpu(X4) @ np.linalg.pinv(Theta_dense, rcond=1e-10)

cores4, last4 = mandy(X4, fpu(X4))
Xt = rng_f.uniform(-0.1, 0.1, (N, 50))
Pt = psi(Xt)
theta_t = Pt[0]
for i in range(1, N):
    theta_t = np.einsum("am,jm->ajm", theta_t, Pt[i]).reshape(-1, Xt.shape[1])
print(f"N = 4: MANDy vs dense pinv, max difference "
      f"{np.abs(mandy_eval(cores4, last4, Xt) - Xi_dense @ theta_t).max():.1e}")

# N = 8: 4^8 = 65,536 candidate functions, never formed
N = 8
X8 = rng_f.uniform(-0.1, 0.1, (N, 400))
t0 = time.perf_counter()
cores8, last8 = mandy(X8, fpu(X8))
Xt = rng_f.uniform(-0.1, 0.1, (N, 200))
err = np.linalg.norm(mandy_eval(cores8, last8, Xt) - fpu(Xt)) / np.linalg.norm(fpu(Xt))
print(f"N = 8: {4 ** N:,d} candidates, {time.perf_counter() - t0:.1f} s, "
      f"bonds {[c.shape[2] for c in cores8]}, test error {err:.1e}")

The bonds grow as $\min(J^k, M)$: the train is exact, and its size is set by the number of
samples, not by $J^N$. The pseudo-inverse gives the minimum-norm least-squares $\Xi$,
not a sparse one — so MANDy predicts well but does not by itself hand you a short
equation. Adding sparsity or a weak formulation inside the TT format is where current
work goes (Houser, Dukic & Bortz 2026, *TT-WSINDy*).

## Wrap-up

* **Same curse, same cure.** DMD's SVD and SINDy's dictionary both blow up with
  dimension; space-time trains and product-dictionary trains remove it.
* **MPS-DMD** reads $U\Sigma V^\dagger$ off the space/time bond: cost logarithmic in both
  $N_x$ and $N_t$, and the input can come straight from a space-time PDE solve.
* **SINDy / MANDy** recover the equation, not just an operator — interpretability is the
  deliverable.
* **Open:** what truncation does to noise, error bars on TN predictions, and going beyond
  $(1+1)$D. For an application that chains POD, DMD and SINDy end to end, see van't Hoff
  et al. (2026) on the atmospheric effects of high-altitude emissions.